<a href="https://colab.research.google.com/github/Nadayok/missing_digit_cnn/blob/main/capstone_digits.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import torch
import sklearn
import numpy
import matplotlib

print("PyTorch:", torch.__version__)
print("scikit-learn:", sklearn.__version__)
print("GPU available:", torch.cuda.is_available())

PyTorch: 2.11.0+cpu
scikit-learn: 1.6.1
GPU available: False


In [6]:
%%writefile config.json
{
  "seed": 42,
  "batch_size": 64,
  "epochs": 80,
  "learning_rate": 0.001,
  "weight_decay": 0.0001,
  "dropout": 0.3,
  "patience": 10,
  "occlusion_sizes": [0, 1, 2, 3]
}

Overwriting config.json


In [7]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import json
import random
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

cfg = json.loads(Path("config.json").read_text())
out = Path("results")
out.mkdir(exist_ok=True)

seed = cfg["seed"]
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
print("Device:", device)

digits = load_digits()

# Normalize ด้วยขอบเขตค่าที่ทราบจากคำอธิบายข้อมูล
# ไม่ได้คำนวณค่าสถิติจาก test
X = (digits.images / 16.0).astype(np.float32)
y = digits.target.astype(np.int64)

indices = np.arange(len(y))

train_idx, remaining_idx = train_test_split(
    indices,
    test_size=0.4,
    random_state=seed,
    stratify=y
)

val_idx, test_idx = train_test_split(
    remaining_idx,
    test_size=0.5,
    random_state=seed,
    stratify=y[remaining_idx]
)

assert not set(train_idx) & set(val_idx)
assert not set(train_idx) & set(test_idx)
assert not set(val_idx) & set(test_idx)

np.savez(
    out / "split_indices.npz",
    train=train_idx,
    val=val_idx,
    test=test_idx
)

for name, idx in [
    ("train", train_idx),
    ("validation", val_idx),
    ("test", test_idx)
]:
    print(name, len(idx))
    print("class counts:", np.bincount(y[idx], minlength=10))

Device: cpu
train 1078
class counts: [107 109 106 110 109 109 109 107 104 108]
validation 359
class counts: [36 36 36 36 36 36 36 36 35 36]
test 360
class counts: [35 37 35 37 36 37 36 36 35 36]


In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(9, 4))

for ax, idx in zip(axes.ravel(), train_idx[:10]):
    ax.imshow(X[idx], cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"Label: {y[idx]}")
    ax.axis("off")

plt.tight_layout()
plt.savefig(out / "train_examples.png", dpi=160)
plt.show()